# Camera-ready deep baselines: San Diego 1&2 (+ TSTTD Pavia re-run)
**Upload-and-run on a GPU runtime.** Runs THANTD, HTD-Net, and TSTTD on the
ARCHIVE-EXACT San Diego protocol (half-open region boxes, GT-mean signature at
median test norm, pool-based 10% planting — asserted against the MLSP archive),
plus a TSTTD re-run on the Pavia scn4 protocol that archives raw scores.
Checkpoint-resume per (model, scene, seed); ends with a zip auto-download.
Runtime ~1.5-2.5h on a T4.


In [ ]:
!git clone -b tsp-repro --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, torch
sys.path.insert(0, '.'); sys.path.insert(0, 'experiments/spatial')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
CKPT_DIR, OUT_DIR = 'ckpt_deep_sd', 'results_deep_sd'
import os; os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# ---- generic archive-protocol runner (SD scenes) ----
import os, json, numpy as np, torch
from sklearn.metrics import roc_auc_score
from colab_deep import sd_protocol as SP

def run_deep_sd(scene_name, det_name, fit_fn, score_fn, out_dir=OUT_DIR, ckpt_dir=CKPT_DIR):
    sc = SP.build(scene_name)
    res = {}
    for seed in SP.SEEDS:
        ck = os.path.join(ckpt_dir, f"{det_name}_{scene_name}_seed{seed}.pt")
        state = fit_fn(sc, seed, ck)
        for mode, th in SP.cells():
            p = os.path.join(out_dir, f"scores_{scene_name}_{det_name}_seed{seed}_{mode}_{th}.npz")
            if os.path.exists(p):
                d = np.load(p); a = roc_auc_score(d["labels"], d["scores"])
            else:
                pl, lab = SP.plant_pool(sc["te"], sc["sig"], th, mode, seed=seed,
                                        spatial_shape=sc["te_shape"])
                s = np.asarray(score_fn(state, sc, pl), np.float32)
                a = roc_auc_score(lab, s)
                np.savez_compressed(p, scores=s, labels=lab)
            res.setdefault(f"{mode}|{th}", []).append(a)
            print(f"[{scene_name}] {det_name} seed{seed} {mode} th={th}: AUC={a:.4f}", flush=True)
    with open(os.path.join(out_dir, f"{scene_name}_{det_name}_results.json"), "w") as f:
        json.dump(res, f, indent=1)
    print(f"=== {det_name} on {scene_name} (5 seeds) ===")
    for k, v in res.items(): print(f"  {k}: {np.mean(v):.3f}+/-{np.std(v):.3f}")
    return res


## THANTD


In [ ]:
# ---- THANTD (paper recipe; bkg pool = secondary pixels) ----
from thantd_model import THANTD, build_thantd_samples, train_thantd, score_thantd

def thantd_fit(sc, seed, ck):
    m = THANTD(b=sc["tr"].shape[1])
    if os.path.exists(ck):
        m.load_state_dict(torch.load(ck, map_location="cpu")); m.to(DEVICE).eval()
        print("  resumed", ck); return m
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    a, p, n = build_thantd_samples(np.asarray(sc["tr"]), sc["sig"], alpha=0.5,
                                   n_samples=1024, rng=rng, bkg_pool=np.asarray(sc["tr"]))
    m.to(DEVICE)
    train_thantd(m, a, p, n, epochs=300, batch_size=64, lr=1e-4, margin=0.3, device=DEVICE)
    torch.save(m.state_dict(), ck); return m

def thantd_sc(m, sc, planted): return score_thantd(m, sc["sig"], planted, device=DEVICE)

for scn in ("sandiego", "sandiego2"):
    run_deep_sd(scn, "THANTD", thantd_fit, thantd_sc)


## HTD-Net


In [ ]:
# ---- HTD-Net (U-AE -> LP -> ACE labels -> SD-CNN), paper recipe ----
from colab_deep import htdnet_model as H

def htdnet_fit(sc, seed, ck):
    tr, sig = np.asarray(sc["tr"]), sc["sig"]
    if os.path.exists(ck):
        blob = torch.load(ck, map_location="cpu")
        sd = H.SDCNN(); sd.load_state_dict(blob["sdcnn"]); sd._scale = blob["scale"]
        sd.to(DEVICE).eval(); print("  resumed", ck)
        return dict(sd=sd, gen_t=blob["gen_t"], bkg=blob["bkg"])
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    uae, s_ = H.train_uae(tr, epochs=150, device=DEVICE, seed=seed)
    gen_t = H.generate_targets(uae, s_, sig, tr, n_samples=1000, device=DEVICE, rng=rng)
    bkg = H.lp_background_selection(tr, sig, n_direct=60, n_total=500)
    sd = H.train_sdcnn(gen_t, bkg, H.ACELabeler(tr), epochs=30, pairs_per_epoch=100_000,
                       device=DEVICE, seed=seed, log_every=10)
    torch.save(dict(sdcnn=sd.state_dict(), scale=sd._scale, gen_t=gen_t, bkg=bkg), ck)
    return dict(sd=sd, gen_t=gen_t, bkg=bkg)

def htdnet_sc(st, sc, planted):
    return H.htdnet_detect(st["sd"], planted, st["gen_t"], st["bkg"], device=DEVICE)

for scn in ("sandiego", "sandiego2"):
    run_deep_sd(scn, "HTDNet", htdnet_fit, htdnet_sc)


## TSTTD — San Diego 1&2


In [ ]:
# ---- TSTTD (vendor code verbatim; our secondary pixels as the bkg pool) ----
import sys
sys.path.insert(0, "/content/repo/colab_deep/vendor_tsttd")
os.chdir("/content/repo/colab_deep/vendor_tsttd")
import Train_eval as TE
from Train_eval import spectral_group
from Model import SpectralGroupAttention
os.chdir("/content/repo")

def make_tsttd(scene_name):
    sc = SP.build(scene_name)
    BAND = sc["tr"].shape[1]                                # 189
    mn, mx = sc["data"].min(), sc["data"].max()
    S = lambda x: ((x - mn) / (mx - mn)).astype(np.float32)  # vendor standard(), cube-consistent

    class OurData(torch.utils.data.Dataset):                # vendor recipe; bkg = secondary px
        def __init__(self, path=None):
            bkg = S(np.asarray(sc["tr"])); ts = S(sc["sig"])[None, :]
            al = np.random.uniform(0, 0.1, (len(bkg), 1)).astype(np.float32)
            self.target_samples, self.background_samples = al*bkg + (1-al)*ts, bkg
            self.target_spectrum, self.nums = ts, len(bkg)
        def __getitem__(self, i): return self.target_samples[i], self.background_samples[i]
        def __len__(self): return self.nums

    def fit(sc_, seed, ck):
        cfg = {"state":"train","epoch":20,"band":BAND,"multiplier":2,"seed":seed,"batch_size":64,
               "group_length":20,"depth":4,"heads":4,"dim_head":64,"mlp_dim":64,"adjust":False,
               "channel":128,"lr":1e-4,"epision":5,"grad_clip":1.,"device":"cuda:0",
               "training_load_weight":None,"save_dir":f"./Ckpt_{scene_name}_s{seed}/",
               "test_load_weight":None,"path":"ours"}
        model = SpectralGroupAttention(band=BAND, m=cfg["group_length"], d=cfg["channel"],
                                       depth=cfg["depth"], heads=cfg["heads"],
                                       dim_head=cfg["dim_head"], mlp_dim=cfg["mlp_dim"],
                                       adjust=cfg["adjust"]).cuda()
        if os.path.exists(ck):
            model.load_state_dict(torch.load(ck, map_location="cuda")); model.eval()
            print("  resumed", ck); model._cfg = cfg; return model
        np.random.seed(seed); TE.Data = OurData
        cwd = os.getcwd(); os.chdir("/content/repo/colab_deep/vendor_tsttd")
        try:
            os.makedirs(cfg["save_dir"] + "/ours/", exist_ok=True)
            TE.train(cfg)
        finally: os.chdir(cwd)
        ckdir = "/content/repo/colab_deep/vendor_tsttd/" + cfg["save_dir"].lstrip("./") + "/ours/"
        last = max(os.listdir(ckdir), key=lambda s: int("".join(filter(str.isdigit, s)) or -1))
        model.load_state_dict(torch.load(ckdir + last)); model.eval()
        torch.save(model.state_dict(), ck); model._cfg = cfg
        return model

    def score(model, sc_, planted):
        cfg = model._cfg
        tf = model(spectral_group(S(sc_["sig"])[None, :], BAND, cfg["group_length"]).cuda()).detach()
        X, out = S(planted), []
        for i in range(0, len(X), 512):
            f = model(spectral_group(X[i:i+512], BAND, cfg["group_length"]).cuda()).detach()
            out.append(torch.nn.functional.cosine_similarity(f, tf.expand_as(f), -1).cpu().numpy())
        return np.concatenate(out)

    return fit, score

for scn in ("sandiego", "sandiego2"):
    fit, score = make_tsttd(scn)
    run_deep_sd(scn, "TSTTD", fit, score)


## TSTTD — Pavia scn4 re-run


In [ ]:
# ---- TSTTD on the Pavia scn4 protocol (RE-RUN, raw scores archived this time) ----
# Same planting as the other Pavia deep rows (paper_protocol, 10% of ALL test px).
from colab_deep import paper_protocol as P
proto = P.build_protocol()          # tr 4026px, te 7476px, foreign bitumen sig
BANDP = proto["tr"].shape[1]        # 103
mnp, mxp = proto["data"].min(), proto["data"].max()
Sp = lambda x: ((x - mnp) / (mxp - mnp)).astype(np.float32)

class PaviaData(torch.utils.data.Dataset):
    def __init__(self, path=None):
        bkg = Sp(proto["tr"]); ts = Sp(proto["sig"])[None, :]
        al = np.random.uniform(0, 0.1, (len(bkg), 1)).astype(np.float32)
        self.target_samples, self.background_samples = al*bkg + (1-al)*ts, bkg
        self.target_spectrum, self.nums = ts, len(bkg)
    def __getitem__(self, i): return self.target_samples[i], self.background_samples[i]
    def __len__(self): return self.nums

res_pv = {}
for seed in SP.SEEDS:
    cfg = {"state":"train","epoch":20,"band":BANDP,"multiplier":2,"seed":seed,"batch_size":64,
           "group_length":20,"depth":4,"heads":4,"dim_head":64,"mlp_dim":64,"adjust":False,
           "channel":128,"lr":1e-4,"epision":5,"grad_clip":1.,"device":"cuda:0",
           "training_load_weight":None,"save_dir":f"./Ckpt_pavia_s{seed}/",
           "test_load_weight":None,"path":"ours"}
    ck = os.path.join(CKPT_DIR, f"TSTTD_pavia4_seed{seed}.pt")
    model = SpectralGroupAttention(band=BANDP, m=20, d=128, depth=4, heads=4,
                                   dim_head=64, mlp_dim=64, adjust=False).cuda()
    if os.path.exists(ck):
        model.load_state_dict(torch.load(ck, map_location="cuda")); model.eval()
        print("  resumed", ck)
    else:
        np.random.seed(seed); TE.Data = PaviaData
        cwd = os.getcwd(); os.chdir("/content/repo/colab_deep/vendor_tsttd")
        try:
            os.makedirs(cfg["save_dir"] + "/ours/", exist_ok=True)
            TE.train(cfg)
        finally: os.chdir(cwd)
        ckdir = "/content/repo/colab_deep/vendor_tsttd/" + cfg["save_dir"].lstrip("./") + "/ours/"
        last = max(os.listdir(ckdir), key=lambda s: int("".join(filter(str.isdigit, s)) or -1))
        model.load_state_dict(torch.load(ckdir + last)); model.eval()
        torch.save(model.state_dict(), ck)
    def sc_pv(planted):
        tf = model(spectral_group(Sp(proto["sig"])[None, :], BANDP, 20).cuda()).detach()
        X, out = Sp(planted), []
        for i in range(0, len(X), 512):
            f = model(spectral_group(X[i:i+512], BANDP, 20).cuda()).detach()
            out.append(torch.nn.functional.cosine_similarity(f, tf.expand_as(f), -1).cpu().numpy())
        return np.concatenate(out)
    for mode, th in SP.cells():
        p = os.path.join(OUT_DIR, f"scores_pavia4_TSTTD_seed{seed}_{mode}_{th}.npz")
        if os.path.exists(p):
            d = np.load(p); a = roc_auc_score(d["labels"], d["scores"])
        else:
            pl, lab = P.plant_targets(proto["te"], proto["sig"], th, 0.10, model=mode,
                                      seed=seed, spatial_shape=proto["te_shape"], edge_guard=3)
            s = np.asarray(sc_pv(pl), np.float32)
            a = roc_auc_score(lab, s); np.savez_compressed(p, scores=s, labels=lab)
        res_pv.setdefault(f"{mode}|{th}", []).append(a)
        print(f"[pavia4] TSTTD seed{seed} {mode} th={th}: AUC={a:.4f}", flush=True)
import json as _json
with open(os.path.join(OUT_DIR, "pavia4_TSTTD_results.json"), "w") as f:
    _json.dump(res_pv, f, indent=1)
for k, v in res_pv.items(): print(f"  {k}: {np.mean(v):.3f}+/-{np.std(v):.3f}")


## Zip + download


In [ ]:
import zipfile
with zipfile.ZipFile('deep_sd_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for d in (OUT_DIR, CKPT_DIR):
        for root, _, files in os.walk(d):
            for fn in files: z.write(os.path.join(root, fn))
print('zipped -> deep_sd_results.zip')
try:
    from google.colab import files; files.download('deep_sd_results.zip')
except Exception: print('(not on Colab)')
